In [0]:
from pyspark.sql.functions import replace,expr,substring
#cleansing data remove duplicates and null values
df_customers=df_customers.dropDuplicates()
#removing invalid feedback score
df_customers=df_customers.filter((df_customers['feedback_score']>=1)| (df_customers['feedback_score']<=10))
df_customers=df_customers.withColumn('email',expr("concat(substring(email,1,length(email)-3),'com')"))
display(df_customers)

In [0]:
from pyspark.sql.functions import to_date,date_format
df_products=df_products.dropDuplicates()
df_products=df_products.dropna()
df_products=df_products.withColumn('restock_date',date_format('restock_date','dd-MMM-yyyy'))
df_products = df_products.withColumn(
    'restock_date',
    to_date('restock_date', 'dd-MMM-yyyy'))



In [0]:
#df_retailer_sales
df_retailer_sales=df_retailer_sales.dropDuplicates()
df_retailer_sales=df_retailer_sales.dropna()
df_retailer_sales.printSchema()



In [0]:
# Joining data frames 
df_customer_sales=df_retailer_sales.join(df_customers,df_retailer_sales.customer_id==df_customers.customer_id,'left').select(df_retailer_sales['transaction_id'],df_retailer_sales['customer_id'],df_retailer_sales['product_id'],df_retailer_sales['quantity'],df_retailer_sales['price'],df_retailer_sales['transaction_date'],df_customers['gender'],df_customers['age'],df_customers['feedback_score'])
display(df_customer_sales)
#df_customer_sales.count()

In [0]:
df_customer_sales.count()

In [0]:
df_final_merge=df_customer_sales\
.join(df_products,df_customer_sales.product_id==df_products.product_id,'inner')\
.select(df_customer_sales['*'],df_products['product_name'],df_products['category'],df_products['stock_quantity'],df_products['supplier_name'],df_products['restock_date'])
display(df_final_merge.limit(10))

In [0]:
from pyspark.sql.functions import sum,desc
from pyspark.sql.types import IntegerType
df_final_merge=df_final_merge.withColumn('price',df_final_merge['price'].cast(IntegerType()))
df_final_merge=df_final_merge.withColumn('PurchaseAmount',df_final_merge['quantity']*df_final_merge['price'])
df_final_merge_customer_agg=df_final_merge.groupBy('customer_id').agg(sum('PurchaseAmount').alias('TotalPurchase')).orderBy(desc('TotalPurchase'))
display(df_final_merge_customer_agg.limit(5))

In [0]:

from pyspark.sql.functions import when,col,avg,count,round
df_final_merge=df_final_merge.withColumn('AgeCategory',when(col('age')<=30,'Young').when (col('age')<=60,'Middle_Age').otherwise('Old_Age'))
#avg feedback by gender
df_final_merge_gender_feedback=df_final_merge.groupBy('gender').agg(round(avg('feedback_score'),2).alias('Avg_Feedback_Score')).orderBy(desc('Avg_Feedback_Score'))
#display(df_final_merge_gender_feedback)
#product with low stock and what is the recent restock dates
df_final_merge_low_stock=df_final_merge.filter(df_final_merge['stock_quantity']<100).select(df_final_merge['product_id'],df_final_merge['category'],df_final_merge['supplier_name'],df_final_merge['stock_quantity'],df_final_merge['restock_date']).orderBy('stock_quantity')
#Analyze the product category based on less stock quantity
df_final_merge_low_stock_categoty=df_final_merge_low_stock.groupBy('category').agg(count('product_id').alias('count')).orderBy(desc('count'))
display(df_final_merge_low_stock_categoty.limit(10))



In [0]:
#monthly purchase amount 
from pyspark.sql.functions import month,sum,col,desc
df_final_merge=df_final_merge.withColumn('Month',month('transaction_date'))
df_final_merge_month_sales=df_final_merge.groupBy(month('transaction_date')).agg(sum('PurchaseAmount').alias('MonthPurchase')).orderBy(desc('MonthPurchase'))
display(df_final_merge_month_sales)

In [0]:
#Calculate average purchase frequency per customer.
df_final_merge_customer_count=df_final_merge.groupBy('customer_id','Month').agg(count('customer_id').alias('ascount'))
df_final_merge_customer_count=df_final_merge_customer_count.groupBy('customer_id').agg(avg('ascount'))
#df_final_merge_customer_count.show()



# Product Performance & Inventory Optimization

In [0]:
display(df_final_merge)

In [0]:
#high demand and low inventory products
from pyspark.sql.functions import sum
df_final_merge_products=df_final_merge.groupBy('product_id','product_name').agg(sum('quantity').alias('sum_quantity'),sum('stock_quantity').alias('stock_quantity'))
df_final_merge_products=df_final_merge_products.filter((df_final_merge_products['sum_quantity']>=40) & (df_final_merge_products['stock_quantity']<1000))                                                           
display(df_final_merge_products)


In [0]:
# Avg feedback based on age category
Age_category_feedback=df_final_merge.groupBy('AgeCategory').agg(avg('feedback_score'))
display(Age_category_feedback)

In [0]:
#unsatisfied valueble customers
from pyspark.sql.functions import sum,avg
valuble_customers=df_final_merge.groupBy('customer_id').agg(avg('feedback_score').alias('avg_feedback'),sum('PurchaseAmount').alias('total_purchase'))
valuble_customers=valuble_customers.filter((valuble_customers['avg_feedback']<=5 ) & (valuble_customers['total_purchase']>1000))
#display(valuble_customers)

In [0]:
#Correlate feedback score with purchase frequency and amount.
from pyspark.sql.functions import sum,avg,count,desc,asc
feedbackscore=df_final_merge.groupBy('feedback_score').agg(count('transaction_id').alias('NoOfTransactions'),sum('PurchaseAmount').alias('totalpurchaseamount')).orderBy(asc('NoOfTransactions'),desc('totalpurchaseamount'))
display(feedbackscore)

#  Window Function Use Cases

In [0]:
#Rank customers by purchase amount within each month.
from pyspark.sql.window import Window
from pyspark.sql.functions import *
cust_rank=Window.partitionBy('Month').orderBy(desc('PurchaseAmount'))
customers_monthlyrank=df_final_merge.withColumn('customer_rank',rank().over(cust_rank))
customers_monthlyrank_limit=customers_monthlyrank.filter(customers_monthlyrank['customer_rank']<=2).select('customer_id','category','PurchaseAmount','Month')
display(customers_monthlyrank_limit)

In [0]:
#df_final_merge.write.format('delta').saveAsTable('workspace.sales_0915.Golden_Sales_Fact')

In [0]:
#running total amount of customers
from pyspark.sql.window import Window
from pyspark.sql.functions import *
customer_partition=Window.partitionBy('customer_id').orderBy('transaction_date')
df_final_merge_Customer_Cumlative=df_final_merge.withColumn('Cum_Amount',sum('PurchaseAmount').over(customer_partition))
df_final_merge_Customer_Cumlative=df_final_merge_Customer_Cumlative.select('customer_id','transaction_date','PurchaseAmount','Cum_Amount').orderBy('customer_id')
display(df_final_merge_Customer_Cumlative)

In [0]:
#Identify first and last purchase date per customer
from pyspark.sql.window import Window
from pyspark.sql.functions import *
cust_dates=Window.partitionBy(col('customer_id'))
df_final_merge_cust_min_max_purchasedate=df_final_merge.withColumn('Cust_Min_Purchase_Date',min('transaction_date').over(cust_dates)).withColumn('Cust_Max_Purchase_Date',max('transaction_date').over(cust_dates))
df_final_merge_cust_min_max_purchasedate=df_final_merge_cust_min_max_purchasedate.select('customer_id','Cust_Min_Purchase_Date','Cust_Max_Purchase_Date').distinct().orderBy('customer_id')
display(df_final_merge_cust_min_max_purchasedate)